[Back to Data Structures and Algorithms guideline](Data-Structure&Algorithm.html)


## **Backtracking and State-Space Search** {#backtracking-and-state-space-search}

**Backtracking** explores a space of decisions depth first. It extends one partial candidate, abandons that branch as soon as it cannot lead to a required answer, restores the earlier state, and tries the next choice. It is useful when the output is a construction rather than merely a number, when constraints interact across choices, or when no compact polynomial-time formulation is known.

The word "backtracking" describes more than recursion. A correct implementation needs:

- a state containing every fact that affects future legality;
- a complete candidate generator so no valid solution is omitted;
- reversible state updates so sibling branches remain independent;
- safe pruning predicates that reject only branches with no useful descendant;
- a clear output contract: find one solution, enumerate all, count them, or optimize an objective.

Backtracking often has exponential worst-case time because it may represent an exponential number of valid or potentially valid constructions. Its practical power comes from reducing the explored tree through constraints, canonical ordering, propagation, good branching choices, and objective bounds.

::: {.callout-important}
Backtracking does not "undo a wrong answer." It restores a partial state after completely handling one branch, whether that branch failed or produced a valid solution. The restoration is what makes the next sibling branch correct.
:::


### **State-Space Trees** {#state-space-trees}

A **state-space tree** is a conceptual tree of partial decisions. The root is the empty state, an edge adds one legal choice, and a node represents the resulting partial assignment. Leaves may be complete solutions, complete non-solutions, or dead ends discovered before maximum depth.

The tree is normally generated lazily. The algorithm stores only the current root-to-node path and enough metadata to produce children; it does not allocate every node in advance. Depth-first traversal is therefore a natural fit because returning from a recursive call corresponds to moving back to the parent state.

If every internal node has at most branching factor $b$ and solutions have depth at most $d$, a full tree contains

$$
1+b+b^2+\cdots+b^d=\frac{b^{d+1}-1}{b-1}=O(b^d)
$$

nodes when $b>1$. Here $b$ is the maximum number of candidate choices per state and $d$ is the number of decisions on a complete path. This is an upper bound: constraints can reduce the effective branching factor dramatically. Output size can also force exponential work; enumerating all $2^n$ subsets cannot take polynomial total time because the output itself is exponential.

~~~text
SEARCH(state, path)
    if state is a complete valid solution
        record a copy of path
        return
    if state cannot be completed
        return

    for each legal choice from state
        apply choice to state and path
        SEARCH(updated state, path)
        undo choice from state and path
~~~

![A regular backtracking traversal visits one depth-first branch, returns to the latest decision, and continues with its next sibling.](assets/backtracking-state-space-tree.svg){fig-align="center" width="42%"}

*Visual source: [Backtracking without backjumping](https://commons.wikimedia.org/wiki/File:Backtracking-no-backjumping.svg), public domain. The unchanged SVG is stored locally for reliable rendering.*

<details>
<summary>Python implementation: materialize the leaves of a binary state-space tree</summary>

~~~python
def enumerate_binary_assignments(length: int) -> tuple[list[tuple[int, ...]], int]:
    """Return all binary leaves and the number of tree nodes visited."""
    if length < 0:
        raise ValueError("length must be non-negative")

    path: list[int] = []
    leaves: list[tuple[int, ...]] = []
    visited_nodes = 0

    def search(depth: int) -> None:
        nonlocal visited_nodes
        visited_nodes += 1  # Count the current partial assignment as one node.

        if depth == length:
            leaves.append(tuple(path))
            return

        for bit in (0, 1):
            path.append(bit)
            search(depth + 1)
            path.pop()

    search(0)
    return leaves, visited_nodes


assignments, nodes = enumerate_binary_assignments(3)
assert len(assignments) == 8
assert nodes == 15  # 1 + 2 + 4 + 8 nodes in a full depth-3 tree.
print(assignments)
print("visited nodes:", nodes)
~~~

</details>

The full binary tree has $2^{n+1}-1$ visited nodes. Copying each length-$n$ leaf makes total output time $\Theta(n2^n)$ and output space $\Theta(n2^n)$. Excluding the output, depth-first search uses $O(n)$ path and call-stack space rather than storing the entire tree.

**Practice:** [LeetCode 78 - Subsets](https://leetcode.com/problems/subsets/) asks you to map include/exclude decisions to the leaves or intermediate nodes of a state-space tree.


### **Choose, Explore, and Unchoose** {#choose-explore-and-unchoose}

The canonical backtracking frame has three operations:

1. **Choose:** mutate the current path and every auxiliary structure affected by the candidate.
2. **Explore:** recurse while the chosen candidate is part of the state.
3. **Unchoose:** reverse all mutations in the opposite order before trying the next sibling.

This creates a frame invariant: immediately before each loop iteration, the path and auxiliary state describe exactly the same parent node. If a choice appends an item, marks it used, changes a board cell, and updates a running total, the cleanup must restore all four components.

~~~text
BACKTRACK(parent state)
    for choice in candidates(parent state)
        path.append(choice)          // choose
        mark choice as used

        BACKTRACK(child state)       // explore

        mark choice as unused        // unchoose
        path.pop()
~~~

![The recursive call temporarily sees the chosen item; cleanup restores the exact parent state before the next sibling.](assets/choose-explore-unchoose.svg){fig-align="center" width="100%"}

Two implementation errors are especially common. First, appending <code>path</code> itself to the results stores several references to one mutable list; append <code>path.copy()</code> or an immutable tuple. Second, an early <code>return</code> between choose and unchoose can skip cleanup. Keep cleanup adjacent to the recursive call, or use a structure that guarantees restoration when control exits unexpectedly.

Permutations illustrate the pattern clearly. At depth $i$, the path contains exactly $i$ distinct items, and <code>used[j]</code> is true exactly when item $j$ appears in that path.

<details>
<summary>Python implementation: permutations with explicit reversible state</summary>

~~~python
from collections.abc import Sequence
from typing import TypeVar

T = TypeVar("T")


def permutations(items: Sequence[T]) -> list[tuple[T, ...]]:
    """Return permutations of distinct items using choose/explore/unchoose."""
    if len(set(items)) != len(items):
        raise ValueError("this basic template requires distinct hashable items")

    path: list[T] = []
    used = [False] * len(items)
    results: list[tuple[T, ...]] = []

    def search() -> None:
        if len(path) == len(items):
            results.append(tuple(path))  # Freeze the current solution.
            return

        for index, item in enumerate(items):
            if used[index]:
                continue

            path.append(item)       # Choose.
            used[index] = True
            search()                # Explore.
            used[index] = False     # Unchoose in reverse order.
            path.pop()

    search()
    return results


answer = permutations(["A", "B", "C"])
assert len(answer) == 6
assert ("C", "B", "A") in answer
print(answer)
~~~

</details>

For $n$ distinct items, the search has $n!$ leaves. Producing each length-$n$ permutation requires $O(n)$ copying, so output-sensitive time is $\Theta(n\cdot n!)$ and output space is the same order. Excluding output, the path, used array, and recursion stack use $O(n)$ space.

**Practice:** [LeetCode 46 - Permutations](https://leetcode.com/problems/permutations/) directly tests reversible path and used-set invariants.


### **Pruning Strategies** {#pruning-strategies}

**Pruning** stops exploring a partial state after proving that no descendant can contribute to the requested output. A prune changes performance, not the answer set. "This branch looks unlikely" is a heuristic ordering decision, not a valid prune.

Four common forms are:

| Pruning form | Safe condition | Example |
|---|---|---|
| Feasibility pruning | a monotone constraint is already violated | closing parentheses exceed opening parentheses |
| Branch and bound | an optimistic bound cannot beat the incumbent | maximum possible remaining value is too small |
| Symmetry breaking | another canonical branch represents the same constructions | do not place the same value into equivalent empty bins |
| Constraint propagation | reducing domains proves some variable has no legal value | an empty Sudoku candidate set |

For maximization, let $B$ be the best complete objective found so far and $UB(s)$ an upper bound on every completion of state $s$. The branch is safely pruned when

$$
UB(s)\le B.
$$

The bound must be optimistic: it may overestimate what can be achieved, but must never underestimate it. A tighter bound prunes more; a cheaper bound costs less to compute.

Candidate ordering is complementary. Trying promising choices first can find a strong incumbent early, making later branch-and-bound checks more effective. In constraint satisfaction, the **minimum remaining values (MRV)** rule chooses the variable with the smallest current domain, exposing failure early. Ordering affects runtime but not correctness if all remaining choices are still explored.

~~~text
GENERATE-PARENTHESES(open, close, path)
    if open = n and close = n
        record path
        return
    if open < n
        choose "("; recurse; unchoose
    if close < open
        choose ")"; recurse; unchoose
~~~

![Feasibility, objective bounds, symmetry, and propagation reject branches for different correctness reasons.](assets/pruning-strategies.svg){fig-align="center" width="100%"}

The condition <code>close &lt; open</code> is a feasibility prune. Once a prefix has more closing than opening parentheses, adding later characters cannot repair that prefix. Likewise, no branch may use more than $n$ opening parentheses.

<details>
<summary>Python implementation: generate parentheses with feasibility pruning</summary>

~~~python
def generate_parentheses(pair_count: int) -> list[str]:
    """Generate every balanced string containing pair_count pairs."""
    if pair_count < 0:
        raise ValueError("pair_count must be non-negative")

    path: list[str] = []
    results: list[str] = []

    def search(opened: int, closed: int) -> None:
        if opened == pair_count and closed == pair_count:
            results.append("".join(path))
            return

        if opened < pair_count:
            path.append("(")
            search(opened + 1, closed)
            path.pop()

        # This guard prevents every prefix with too many closing brackets.
        if closed < opened:
            path.append(")")
            search(opened, closed + 1)
            path.pop()

    search(0, 0)
    return results


answer = generate_parentheses(3)
assert len(answer) == 5
assert set(answer) == {"((()))", "(()())", "(())()", "()(())", "()()()"}
print(answer)
~~~

</details>

The number of valid outputs is the Catalan number

$$
C_n=\frac{1}{n+1}\binom{2n}{n}.
$$

Every output has length $2n$, so output-sensitive time is $\Theta(nC_n)$ and output space is the same order. The path and recursion stack use $O(n)$ auxiliary space. Feasibility pruning avoids most of the $2^{2n}$ unrestricted binary strings but cannot avoid constructing the valid outputs.

**Practice:** [LeetCode 22 - Generate Parentheses](https://leetcode.com/problems/generate-parentheses/) is a direct exercise in proving prefix pruning safe.


### **Subsets, Permutations, and Combinations** {#subsets-permutations-and-combinations}

These outputs differ in whether order matters and whether every item must be used. Their backtracking states should encode that distinction directly.

- A **subset** may contain any selection. A start index generates elements in increasing input order, so the same subset is not produced under different orders.
- A size-$k$ **combination** is a subset with a target length. The branch can stop when too few items remain to reach $k$.
- A **permutation** uses every selected item in an ordered arrangement. A used set or in-place swap records which positions remain available.

~~~text
SUBSETS(start, path)
    record path
    for i <- start to n-1
        choose items[i]
        SUBSETS(i+1, path)
        unchoose items[i]

COMBINATIONS(start, path, k)
    if length(path) = k: record path; return
    for i over indices leaving enough remaining items
        choose items[i]; recurse with i+1; unchoose

PERMUTATIONS(path, used)
    if length(path) = n: record path; return
    for each unused index i
        mark i; choose items[i]; recurse; unchoose; unmark i
~~~

![Subsets use include/exclude or increasing indices, combinations retain a start index, and permutations retain a used set.](assets/combinatorial-generation.svg){fig-align="center" width="100%"}

Duplicate input values require an additional canonical rule. Sort the input, then skip equal values at the **same recursion depth**. For unique permutations, an equal value at index $i$ may be chosen only after the identical predecessor at $i-1$ has been used in the current path. This keeps one representative ordering while preserving valid uses of repeated values at different depths.

<details>
<summary>Python implementations: subsets, fixed-size combinations, and unique permutations</summary>

~~~python
def all_subsets(items: list[int]) -> list[tuple[int, ...]]:
    results: list[tuple[int, ...]] = []
    path: list[int] = []

    def search(start: int) -> None:
        results.append(tuple(path))
        for index in range(start, len(items)):
            path.append(items[index])
            search(index + 1)
            path.pop()

    search(0)
    return results


def choose_k(items: list[int], k: int) -> list[tuple[int, ...]]:
    if not 0 <= k <= len(items):
        return []
    results: list[tuple[int, ...]] = []
    path: list[int] = []

    def search(start: int) -> None:
        if len(path) == k:
            results.append(tuple(path))
            return

        needed = k - len(path)
        # The last candidate must leave needed - 1 later elements.
        last_start = len(items) - needed
        for index in range(start, last_start + 1):
            path.append(items[index])
            search(index + 1)
            path.pop()

    search(0)
    return results


def unique_permutations(items: list[int]) -> list[tuple[int, ...]]:
    ordered = sorted(items)
    used = [False] * len(ordered)
    path: list[int] = []
    results: list[tuple[int, ...]] = []

    def search() -> None:
        if len(path) == len(ordered):
            results.append(tuple(path))
            return

        for index, value in enumerate(ordered):
            if used[index]:
                continue
            if index > 0 and value == ordered[index - 1] and not used[index - 1]:
                continue  # Skip the same value at this recursion depth.

            used[index] = True
            path.append(value)
            search()
            path.pop()
            used[index] = False

    search()
    return results


assert len(all_subsets([1, 2, 3])) == 8
assert choose_k([1, 2, 3, 4], 2) == [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
assert set(unique_permutations([1, 1, 2])) == {(1, 1, 2), (1, 2, 1), (2, 1, 1)}
print(all_subsets([1, 2, 3]))
print(choose_k([1, 2, 3, 4], 2))
print(unique_permutations([1, 1, 2]))
~~~

</details>

| Output family | Number for distinct $n$ items | Time including copies | Auxiliary path state |
|---|---:|---:|---:|
| All subsets | $2^n$ | $\Theta(n2^n)$ | $O(n)$ |
| Size-$k$ combinations | $\binom{n}{k}$ | $\Theta(k\binom{n}{k})$ | $O(k)$ |
| All permutations | $n!$ | $\Theta(n\cdot n!)$ | $O(n)$ |

These bounds are output-sensitive. No algorithm can explicitly return all results faster than the size required to write them.

**Practice:** [LeetCode 90 - Subsets II](https://leetcode.com/problems/subsets-ii/) tests same-depth duplicate skipping without deleting legitimate repeated values.


### **Constraint and Board Search** {#constraint-and-board-search}

A **constraint satisfaction problem (CSP)** has variables, a domain of candidate values for each variable, and constraints relating assignments. Backtracking selects an unassigned variable, tries a value consistent with the current assignment, propagates its effects, and restores domains when returning.

Board-search problems encode the same pattern spatially:

- N-Queens assigns one column to each row while enforcing column and diagonal constraints;
- Sudoku assigns digits to cells and propagates row, column, and box restrictions;
- Word Search assigns a board cell to each string position and temporarily marks cells used by the current path.

For a queen at row $r$ and column $c$, all cells on the same descending diagonal share $r-c$, and all cells on the same ascending diagonal share $r+c$. Sets of occupied columns, $r-c$, and $r+c$ therefore turn each safety test into expected $O(1)$ membership work.

~~~text
N-QUEENS(row r)
    if r = n
        record board
        return
    for column c <- 0 to n-1
        if c, r-c, and r+c are all unused
            mark column and both diagonals
            place queen at (r,c)
            N-QUEENS(r+1)
            remove queen and all three marks
~~~

![The Eight Queens animation shows placement, conflict detection, rollback, and continuation with the next column.](assets/n-queens-backtracking.gif){fig-align="center" width="48%"}

*Visual source: [Hsilgneymerej, Eight-queens animation](https://commons.wikimedia.org/wiki/File:Eight-queens-animation.gif), public domain. The unchanged 114-frame animation is stored locally for reliable rendering.*

<details>
<summary>Python implementation: solve N-Queens with constant-time constraint sets</summary>

~~~python
def solve_n_queens(size: int) -> list[tuple[str, ...]]:
    """Return every size-by-size N-Queens board."""
    if size < 0:
        raise ValueError("size must be non-negative")

    columns: set[int] = set()
    descending: set[int] = set()  # row - column
    ascending: set[int] = set()   # row + column
    queen_columns: list[int] = []
    solutions: list[tuple[str, ...]] = []

    def search(row: int) -> None:
        if row == size:
            board = tuple(
                "." * column + "Q" + "." * (size - column - 1)
                for column in queen_columns
            )
            solutions.append(board)
            return

        for column in range(size):
            down_key = row - column
            up_key = row + column
            if column in columns or down_key in descending or up_key in ascending:
                continue

            queen_columns.append(column)
            columns.add(column)
            descending.add(down_key)
            ascending.add(up_key)

            search(row + 1)

            ascending.remove(up_key)
            descending.remove(down_key)
            columns.remove(column)
            queen_columns.pop()

    search(0)
    return solutions


solutions = solve_n_queens(4)
assert len(solutions) == 2
print("\n\n".join("\n".join(board) for board in solutions))
~~~

</details>

The row-by-row formulation tries at most one queen per column, so a coarse worst-case bound is $O(n!)$ search nodes before diagonal pruning, plus $O(n^2)$ time to materialize each solution board. The active path and three constraint sets use $O(n)$ auxiliary space; returned boards dominate output space.

For harder CSPs, combine backtracking with MRV variable selection, forward checking, bit-mask domains, and propagation of forced assignments. Every propagated mutation must also be reversible or recorded on an undo stack.

**Practice:** [LeetCode 51 - N-Queens](https://leetcode.com/problems/n-queens/) directly exercises row variables, diagonal encodings, and reversible constraint sets.


### **Backtracking and Dynamic Programming** {#backtracking-and-dynamic-programming}

Backtracking and DP can start from the same recurrence. The distinction is whether different decision histories can be merged without changing their possible continuations.

Memoization is safe when a state key contains every future-relevant fact. In Target Sum, after assigning signs to the first $i$ values, future choices depend only on $i$ and the current total $t$. Two histories reaching $(i,t)$ have identical remaining numbers and identical continuation counts, so they may share one cached answer.

Memoization is unsafe when omitted path history affects legality. In a simple-path search, <code>(current_vertex)</code> is not sufficient because different visited sets permit different future moves. The visited set must remain in the state, perhaps as a bit mask; if that creates $2^n$ distinct states, DP may still be exponential.

~~~text
TARGET-SUM(i, total)
    if i = n
        return 1 when total = target, otherwise 0
    return TARGET-SUM(i+1, total + values[i])
         + TARGET-SUM(i+1, total - values[i])

MEMOIZED-TARGET-SUM(i, total)
    use the same recurrence
    cache the answer under key (i, total)
~~~

![Backtracking revisits an equivalent index-total state, while memoization merges both histories into one continuation.](assets/backtracking-vs-dp.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: compare uncached and memoized Target Sum</summary>

~~~python
from functools import cache


def compare_target_sum_calls(
    values: list[int], target: int
) -> tuple[int, int, int]:
    """Return number of sign assignments, naive calls, and memoized body calls."""
    naive_calls = 0

    def backtrack(index: int, total: int) -> int:
        nonlocal naive_calls
        naive_calls += 1
        if index == len(values):
            return int(total == target)
        return (
            backtrack(index + 1, total + values[index])
            + backtrack(index + 1, total - values[index])
        )

    memo_calls = 0

    @cache
    def memoized(index: int, total: int) -> int:
        nonlocal memo_calls
        memo_calls += 1  # Runs once per distinct (index, total) state.
        if index == len(values):
            return int(total == target)
        return (
            memoized(index + 1, total + values[index])
            + memoized(index + 1, total - values[index])
        )

    naive_answer = backtrack(0, 0)
    memo_answer = memoized(0, 0)
    assert naive_answer == memo_answer
    return naive_answer, naive_calls, memo_calls


ways, naive_count, memo_count = compare_target_sum_calls([1] * 10, 0)
assert ways == 252
assert naive_count == 2047
assert memo_count == 66
print(ways, naive_count, memo_count)
~~~

</details>

The uncached search has two choices at each of $n$ positions, so it takes $O(2^n)$ time and $O(n)$ stack space. Let $S=\sum_i |v_i|$. Memoization has at most $O(nS)$ reachable <code>(index,total)</code> states because totals lie between $-S$ and $S$, giving $O(nS)$ time and space up to constant factors.

Use memoization when repeated states exist and the requested output can be summarized. If the task must explicitly enumerate every distinct path, caching only counts or feasibility cannot replace constructing those outputs.

**Practice:** [LeetCode 494 - Target Sum](https://leetcode.com/problems/target-sum/) directly tests when a sign-assignment tree can be merged by <code>(index,total)</code>.


### **Comparison and Selection** {#comparison-and-selection}

Backtracking is appropriate when legal constructions must be generated or when path-specific constraints prevent compact state merging. It becomes effective when invalidity appears early, candidate sets are small after propagation, or one solution can be found without enumerating the full tree.

**Branch and Bound** specializes state-space search for optimization. It maintains an incumbent complete solution and computes an optimistic objective bound for each partial state. Unlike feasibility pruning, a bounded state may contain valid completions; it is discarded only because none can improve the incumbent.

For positive values in subset selection, a simple maximization bound is

$$
UB(i,current)=current+\sum_{j=i}^{n-1} value_j.
$$

It assumes every remaining value can be added, even if the capacity would prevent that, so it is optimistic and therefore safe. A fractional-knapsack relaxation gives a tighter but more expensive bound.

~~~text
BRANCH-AND-BOUND(state s)
    if s is infeasible: return
    update incumbent when s is a better feasible solution
    if optimistic_bound(s) <= incumbent: return
    explore promising child first
    explore remaining children
~~~

![A selection workflow distinguishes greedy dominance, DP state merging, constraint backtracking, and objective branch-and-bound.](assets/search-strategy-selection.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: depth-first branch and bound for subset value</summary>

~~~python
def best_subset_under_limit(
    values: list[int], limit: int
) -> tuple[int, list[int]]:
    """Maximize the sum of positive values without exceeding limit."""
    if limit < 0 or any(value <= 0 for value in values):
        raise ValueError("limit must be non-negative and values positive")

    ordered = sorted(values, reverse=True)  # Large choices may improve incumbent early.
    suffix_sum = [0] * (len(ordered) + 1)
    for index in range(len(ordered) - 1, -1, -1):
        suffix_sum[index] = suffix_sum[index + 1] + ordered[index]

    best_total = 0
    best_path: list[int] = []
    path: list[int] = []

    def search(index: int, current: int) -> None:
        nonlocal best_total, best_path

        if current > limit:
            return  # Infeasible; positive future values cannot repair it.
        if current > best_total:
            best_total = current
            best_path = path.copy()
        if index == len(ordered):
            return

        optimistic = current + suffix_sum[index]
        if optimistic <= best_total:
            return  # Even taking everything cannot beat the incumbent.

        value = ordered[index]
        path.append(value)
        search(index + 1, current + value)  # Try promising include branch first.
        path.pop()
        search(index + 1, current)

    search(0, 0)
    return best_total, best_path


total, chosen = best_subset_under_limit([8, 7, 6, 5], 13)
assert total == 13
assert sum(chosen) == total
print(total, chosen)
~~~

</details>

Branch and bound still has $O(2^n)$ worst-case time because a weak bound may prune nothing. The depth-first implementation uses $O(n)$ path and call-stack space beyond the sorted input and suffix bounds. Ordering and tight bounds improve explored node count, not the formal worst-case guarantee.

| Structural property | Recommended starting point |
|---|---|
| One locally best choice has a safety proof | greedy |
| Equivalent histories share every future continuation | dynamic programming / memoization |
| Path history changes future legality | backtracking with reversible state |
| Optimization has a cheap optimistic bound | branch and bound |
| Domains can be reduced after assignments | constraint propagation + backtracking |
| Every valid construction must be returned | output-sensitive backtracking |

Before implementation, decide whether the task needs one solution, all solutions, a count, or the best objective. The same state-space tree can support each contract, but its stopping conditions, pruning rules, memoized values, and output costs differ.

**Practice:** [LeetCode 473 - Matchsticks to Square](https://leetcode.com/problems/matchsticks-to-square/) is a synthesis problem involving constrained bins, candidate ordering, capacity pruning, and symmetry between equal side lengths.
